## This notebook us used for OpenFISH decoding, OpenDecode Version 2.0, with following characteristics:

* Inputs are projected one-layer .czi images；
* Include preprocess-registration-spot detection-segmentation-decoding-matrixization five parts
* Above parts have corresponding CODE, can be used to skip/force running. (See bottom for the detailed explaination)
| Procedure | Code |
|:---------|:---------:|
| preprocess | 0 |
| registration | 1 |
| spot detection | 2 |
| segmentation | 3 |
| decoding | 4 |
| matrixization | 5 |

___

<div class="alert alert-block alert-info">
<b>Note：</b>DO NOT set channel shift during the imaging. Adjust the exposing time/intensity carefully, avoiding over-exposing, qiench or shadow. Be sure the imaging channel name is same to the CodeBook</div>

<div class="alert alert-block alert-info">
<b>Note：</b>Please use projected images. It is recommanded to not stitch in advance.  Althrough, this code support tiled/stitched input.</div>

<div class="alert alert-block alert-success">
<b><h2> 1.Fill in the parameters</b></h2> 
</div>

<div class="alert alert-block alert-info">
<b>Note: </b>CodeBook format (comma seperate)：</div>
Support single-color/double-color encoding:

| gene | RO1 | RO2 | Round |
|---------:|---------:|---------:|---------:|
| Cck | AF488 | AF546 | R1 |
| Rgs5 | AF488 | AF594 | R1 |
| Nfib | Cy5 | Cy5 | R1 |
| Slc17a6 | AF546 | AF546 | R4 |

In [3]:
# import sys
# sys.path.append("/path/to/OpenDecode/directory")

from openDecode import OpenDecoder, config

para = config.Para(

    # 1. input coding file path.czi, in dict format. Add more lines if needed.
    filename = {
        'R1': 'demo_data/Demo_20uM_A3_R1_depth.czi',
        'R2': 'demo_data/Demo_20uM_A3_R2_depth.czi',
        # 'R3': '/path/to/R3_depth.czi'
    },
    
    codebook_path = 'demo_data/CodingBook.CSV', # CodeBook path, see demo_data for example
    output_path = 'Result', # Output result, all results will be output in this folder
    anchor_channel = 'AF405', # The channel name used for registeration between cycles.
    
    # Extra channel file path.czi input，fill extra = None or extra = {} if no extra files.
    # CytoRNA must be set with key: CytoRNA. Others can be filled accordingly: such as DIC, IHC...
    extra = {
        'DIC': {
            'filepath': 'demo_data/Demo_20uM_A3_DIC_depth.czi',
            'channel': ['AF405', 'DIC'] # Here, rrelated channels need to be filled
        }
    },
    
    run_deblur = True, # Whether run deblur. Extra files will ignore this parameter.
    run_BaSiC = True, # Whether run BaSiC to correct shadows.
    objective = '20X', # Objective Times. This will affect channel physical shifts. Can be filled with any string as long as corresponding to the key in translation_matrix and pixel scaler.
    threads = 48, # Threads used
    run_basic_clustering = False, # Whether run simple clustering, ignore this parameter when genes > 60, default is True
    
    # This code has two concepts: procedure and progress；progress the processing situation each cycle, each channel，output_path/tmp folder will have a yaml to store progress.
    # Program will run form break point automatically depanding on the yaml. Yaml can also be changed mannully.
    # procedure means the five parts from preprocess to matrixazation. Every part has its own code, can be skipped/forced running.
    # 0: preprocess, 1: stitching&regietration, 2: point detection, 3: segmentation, 4: decoding, 5: matrixization
    # See concrete example lowermost.
    skip_procedure = [],
    force_procedure = [],
    
   # Change below parameters according to your microscopy situtation.
    translation_matrix = {
        "20X":{
            "DAPI": [0.0, 0.0], "AF405": [0.0, 0.0], "AF488": [-0.38+5.38, 0.25], "AF546": [-3.61+5.38,0.12], "AF594": [5.38,0.0], "Cy5": [-2.54+5.38, -0.10], "Cy7": [-2.15+5.38,-0.39], "DIC": [5.38, 0.0]
        },
        "40X":{
            "DAPI": [0.0 ,0.0], "AF405": [0.0, 0.0], "AF488": [-0.44, 0.85], "AF546": [-4.48, -0.03], "AF594": [0.0, 0.0], "Cy5": [-2.68, 0.28], "Cy7": [-1.94, -0.03]
        }
    },
    
    pixel_scaler = {
        '20X': 0.325,
        '40X': 0.1625
    }
    )


OD = OpenDecoder(para)

<div class="alert alert-block alert-success">
<b><h2> 2.程序运行(已列出可以修改的参数)</b></h2> 
</div>

In [4]:
# 0
OD.runPreprocess()
# 1
OD.runRegistration()

# 2
# See https://weigertlab.org/spotiflow/api.html#spotiflow.model.spotiflow.Spotiflow.predict for parameters explaination
OD.runSpotiflow(
    model_name = "hybiss",
    intensity_threshold = {
        'AF488': 100, 'AF546': 300, 'AF594': 400, 'Cy5': 100, 'Cy7': 100
    },
    prob_thresh = 0.40
)

# 3
OD.runSegmentation(
    # Parameters for DAPI segmentation
    cellpose_nuclei_kwargs = {
                            'model_type': 'cyto3',
                            'restore_type': 'deblur_cyto3',
                            'diameter': 25.,
                            'flow_threshold': 0.9,
                            'cellprob_threshold': -2.,
                        },
    # Parameters for CytoRNA segmentation
    cellpose_cyto_kwargs = {
                            'model_type': 'cyto3',
                            'restore_type': 'deblur_cyto3',
                            'diameter': 35.,
                            'flow_threshold': 0.6,
                            'cellprob_threshold': -1.,
                        }
)

# 4
# Parameters are fine-tuned, do not need akterred normally
OD.runDecoding(
    # Training part
    dist_threshold = 1.0, # Minimum pixel considering two points come from one RCP (Rolling circle production)
    prob_threshold = 1.75, # Minimum normalized spotiflow output possibilities's difference considering two points come from one RCP 
    # Predicting part
    prob_diff_threshold = 0.2,  # One point may belong to multiple class after prediction, Top1 class possiblity has to be prob_diff_threshold greater than Top 2 class possibility 
    qv_ratio_threshold = 0.80 # If one point is used by mulitple combination, Top2 qv needs to be samller than than the qv_ratio_threshold * Top1 qv
)

# 5
OD.runMatrixization(
    buffer_radius = 2,  # DAPI segmentation buffer radius, in um
    points_distance_threshold = 17.0, # Distance parameter for spots filtering, only works when there is no segmentation available. Otherwise is the average of cell diameters.
) 

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  5.75it/s]


INFO:spotiflow.model.spotiflow:Loading pretrained model: hybiss


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 29934/29934 [00:03<00:00, 9663.78it/s]
[INFO] (sopa.segmentation.aggregation) Aggregating transcripts over 43205 cells


[########################################] | 100% Completed | 1.30 ss
INFO     The Zarr backing store has been changed from None the new file path: Result/raw_sdata.zarr                


The cell IDs in the table or shapes are not valid Xenium Explorer IDs. They will be replaced by integers starting from 0 in the output files.


<div class="alert alert-block alert-warning">
<b><h3>Example about skipping or forcing execuating procedures.</b></h3>
</div>

| Procedure | Code |
|:---------|:---------:|
| preprocess | 0 |
| registration | 1 |
| spot detection | 2 |
| segmentation | 3 |
| decoding | 4 |
| matrixization | 5 |

___

1.**Say 11 cycles in total. 6 cycles imaged total and you want to decode first, no final expression matrix. And you hope the rest 5 cycles can be filled later. No repeat running.**    
    
    Fill in below the first time running
     filename = {
        'R1': 'Demo_R1_depth.czi',
        'R2': 'Demo_R2_depth.czi',
        'R3': 'Demo_R3_depth.czi',
        'R4': 'Demo_R4_depth.czi',
        'R5': 'Demo_R5_depth.czi',
        'R6': 'Demo_R6_depth.czi',
    },
    ...
    skip_procedure: = [5], # This is optional
    
    Fill in below the second time running
    filename = {
        'R1': 'Demo_R1_depth.czi',
        'R2': 'Demo_R2_depth.czi',
        'R3': 'Demo_R3_depth.czi',
        'R4': 'Demo_R4_depth.czi',
        'R5': 'Demo_R5_depth.czi',
        'R6': 'Demo_R6_depth.czi',
        'R7': 'Demo_R7_depth.czi',
        'R8': 'Demo_R8_depth.czi',
        'R9': 'Demo_R9_depth.czi',
        'R10': 'Demo_R10_depth.czi',
        'R11': 'Demo_R11_depth.czi',
    },
    ...
    skip_procedure: = []
    
2.**Say after decoding, you mannually re-run the spotiflow. You want to re-generate the final result:**     

    skip_procedure = [0,1,2,3],
    force_procedure = [4,5]
    
    *Codes about how to run spotiflow mannualy：
    from openDecode.spotdetection import _run_single_spotiflow
    from spotiflow.model import Spotiflow
    import tifffile
    
    img = tifffile.imread("Demo_R1_Cy7.tif")
    
    df = _run_single_spotiflow(img,
                          model = Spotiflow.from_pretrained('hybiss'),
                          intensity_threshold = 100,
                          prob_thresh = 0.40,
                          n_tiles = (5, 5),
                          min_distance = 1, 
                          exclude_border = True,
                          scale = None,
                          subpix = True,
                          peak_mode ='fast',
                          normalizer = 'auto',
                          verbose = True,
                          device = 'cuda')
                          
    df.to_parquet(Demo_R1_Cy7.parquet)
    
3.**Say after running, you additionally do a IHC staning (aSST), you want to add them in the final result:**     

    extra = {
        'CytoRNA': {
            'filepath': 'Demo_rRNA_depth_new.czi',
            'channel': ['AF405', 'AF546'] 
        },
        "IHC(or any names)": {
            'filepath': 'Demo_SST_depth.czi',
            'channel': ['AF405', 'Cy5'] 
        }
    },
    
and then fill below:

    skip_procedure = [2,3,4],
    force_procedure = [],
    
4.**In conclusion, how to fill the skip/force can be decided as folloing basic logic:**  

    decoding depends on spot detection，so re-run decoding after re-run spot detection  
    matrixization depends on decoding，so re-run matrixization after re-run decoding  

        skip_procedure = [0,1,3],
        force_procedure = [2,4,5]

    matrixization also depends on segmentation，so re-run matrixization after re-run segmentation

        skip_procedure = [0,1,2],
        force_procedure = [3,5]

    Except for filename and extra, to apply any other parameters alteration. Fill the force_procedure. Code DOESN'T detect paramters changing.

<div class="alert alert-block alert-warning">
<b><h3>Some Supplementary Information</b></h3>
</div>

1. If the final generated image contains three channels (e.g., DAPI, CytoRNA, SST), Xenium Explorer will ignore the channel names in the OME.TIFF by default and display them as R, G, B in the software. However, you can use the "add image" function again in the software for easier image viewing, and the image will then display the corresponding names for each channel. The channel names are formatted as extra_key + "_" + channel name. For example: CytoRNA_AF546 or SST_Cy5.

2. The result files contain three folders: Registration, Segmentation, and tmp, along with a spatialdata file (raw_sdata.zarr), a Xenium Explorer file, a Decoded_transcripts.parquet file (which stores decoded point information, including a qv column representing point confidence; qv < 20 indicates the point was flagged based on point-to-point distance), and an adata.h5ad file. The Registration folder contains two subfolders for storing images before and after stitching. Pre-stitched images are saved in npy format, post-stitched images in TIFF format, and Spotiflow results in parquet format. Additionally, 1-3 TIFF images of morphology results will be generated based on the input. The Segmentation folder contains cell segmentation results. The tmp folder contains many intermediate files for checkpoint restarts or data inspection.

3. Raw images undergo a simple clip to remove overexposed points, followed by Richardson-Lucy deblurring using an estimated Gaussian PSF. BaSiC is then run as normal.

4. The image stitching and alignment methods remain unchanged, but the alignment parameters have been adjusted to balance speed and accuracy.

5. Spotiflow is currently the only method retained for spot detection.

6. Spot decoding currently involves two steps: first, select points with particularly high confidence to train a random forest model, then use this model to predict all points. If only single-color encoding is used, the points are directly converted to corresponding genes.

7. Finally, an expression matrix is generated and simple clustering is performed (if the number of genes > 60 or run_basic_clustering = True). The segmentation of CytoRNA and DAPI still follows the principle of merging with CytoRNA as the primary and DAPI as the secondary.

8. **Emphasis again: The program can detect changes in input files but cannot detect changes in other inputs. If there are changes to function parameters, etc., please be sure to adjust the skip_procedure and force_procedure parameters.**

9. If unreasonable skip_procedure and force_procedure are set, the program will issue warnings, so carefully check the output fields.

10. CytoRNA only supports two-channel input; do not set a third channel. If the CytoRNA image contains information from other channels, you can set it as:

        extra = {
            'CytoRNA': {
                'filepath': '/path/to/CytoRNA.czi',
                'channel': ['AF405', 'AF546']  # List the involved channel names here
            },
            "Other": {
                'filepath': '/path/to/CytoRNA.czi',
                'channel': ['AF405', 'AF488']  # Same image, but now only requiring AF405 and AF488
            },
            ...
        }
                

10. *For any other questions, reach to xinyoung_li@genetics.ac.cn*

11. If other input format needed, please Open an issue at github